# Nottingham Forest centre-back results validation

This notebook interprets and validates the Forest centre-back comparison before the method is extended to other positions. It checks provider-role agreement, sampling uncertainty, split-half stability, sensitivity to the 18- and 20-zone layouts, and sensitivity to action selection.

Run `scripts/validate_forest_centre_backs.py` first. The generated tables remain local because they are derived from licensed event data.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'

summary = pd.read_csv(PROCESSED_ROOT / 'forest_cb_validation_summary.csv')
bootstrap = pd.read_csv(PROCESSED_ROOT / 'forest_cb_bootstrap_summary.csv')
split_half = pd.read_csv(PROCESSED_ROOT / 'forest_cb_split_half_summary.csv')
layout_robustness = pd.read_csv(PROCESSED_ROOT / 'centre_back_layout_robustness.csv')
action_sensitivity = pd.read_csv(PROCESSED_ROOT / 'forest_cb_action_family_sensitivity.csv')
zone_interpretation = pd.read_parquet(PROCESSED_ROOT / 'forest_cb_zone_interpretation.parquet')
quality_checks = pd.read_csv(PROCESSED_ROOT / 'forest_cb_validation_quality_checks.csv')

## 1. Reproducibility and quality checks

All interval-order checks should be zero. The expected output contains four Forest centre-backs and two rows per player, one for each layout.

In [2]:
quality_checks

,check,value
0,Validated club,Nottingham Forest
1,Forest centre-backs,4
2,Forest layout summary rows,8
3,Forest zone interpretation rows,152
4,Bootstrap summary rows,8
5,Bootstrap zone interval rows,152
6,Split-half summary rows,8
7,Layout robustness populations,2
8,Action-family sensitivity rows,24
9,Bootstrap interval order failures,0


## 2. Spatial interpretation and provider-role agreement

A positive relative lateral shift means the player's action distribution is shifted toward the displayed left side relative to the leave-one-out league baseline; a negative value means a right shift. This is a description of action locations, not continuous player occupancy.

In [3]:
summary[[
    'player_name', 'layout', 'dominant_provider_role',
    'eligible_appearances', 'similarity_score',
    'relative_lateral_shift', 'spatial_side', 'role_spatial_alignment'
]].round(2)

,player_name,layout,dominant_provider_role,eligible_appearances,similarity_score,relative_lateral_shift,spatial_side,role_spatial_alignment
0,Jair Cunha,positional_20,Right centre-back,3,57.92,-21.29,right-shifted,True
1,Jair Cunha,traditional_18,Right centre-back,3,53.74,-21.67,right-shifted,True
2,Morato,positional_20,Left centre-back,11,73.64,14.02,left-shifted,True
3,Morato,traditional_18,Left centre-back,11,72.24,15.32,left-shifted,True
4,Murillo,positional_20,Left centre-back,23,69.62,16.71,left-shifted,True
5,Murillo,traditional_18,Left centre-back,23,68.01,17.76,left-shifted,True
6,N. Milenković,positional_20,Right centre-back,35,75.12,-10.33,right-shifted,True
7,N. Milenković,traditional_18,Right centre-back,35,75.47,-12.23,right-shifted,True


In [4]:
# The largest player-versus-template zone differences for interpretation.
(zone_interpretation
 .assign(absolute_difference=lambda frame: frame['difference'].abs())
 .sort_values(['player_name', 'layout', 'absolute_difference'], ascending=[True, True, False])
 .groupby(['player_name', 'layout'], as_index=False)
 .head(5)[['player_name', 'layout', 'zone_id', 'zone_name', 'player_share', 'template_share', 'difference']]
 .round(3))

,player_name,layout,zone_id,zone_name,player_share,template_share,difference
14,Jair Cunha,positional_20,15,middle-2-1,0.052,0.166,-0.114
12,Jair Cunha,positional_20,13,middle-1-1,0.021,0.134,-0.113
8,Jair Cunha,positional_20,9,bottom-3,0.147,0.036,0.111
7,Jair Cunha,positional_20,8,bottom-2,0.130,0.040,0.090
6,Jair Cunha,positional_20,7,bottom-1,0.087,0.022,0.065
25,Jair Cunha,traditional_18,6,traditional-06,0.245,0.088,0.157
28,Jair Cunha,traditional_18,9,traditional-09,0.200,0.083,0.118
27,Jair Cunha,traditional_18,8,traditional-08,0.023,0.111,-0.088
24,Jair Cunha,traditional_18,5,traditional-05,0.042,0.115,-0.073
22,Jair Cunha,traditional_18,3,traditional-03,0.108,0.041,0.067


## 3. Sampling uncertainty and within-player stability

Bootstrap intervals resample whole appearances, preserving each match-level zone distribution. The league baseline is held fixed, so these intervals represent uncertainty in the player's appearance sample rather than uncertainty in the league template. Split-half similarity repeatedly divides a player's appearances into two groups and compares the resulting profiles. Jair Cunha is not given a split-half estimate because three appearances are insufficient for a meaningful stability check.

In [5]:
bootstrap[[
    'player_name', 'layout', 'appearance_count', 'similarity_score',
    'similarity_ci_low', 'similarity_ci_high', 'similarity_bootstrap_sd'
]].round(2)

,player_name,layout,appearance_count,similarity_score,similarity_ci_low,similarity_ci_high,similarity_bootstrap_sd
0,Jair Cunha,positional_20,3,57.92,43.45,67.52,6.49
1,Jair Cunha,traditional_18,3,53.74,39.24,66.53,7.44
2,Morato,positional_20,11,73.64,67.47,76.41,2.29
3,Morato,traditional_18,11,72.24,65.78,75.62,2.54
4,Murillo,positional_20,23,69.62,66.26,71.85,1.40
5,Murillo,traditional_18,23,68.01,65.12,70.80,1.48
6,N. Milenković,positional_20,35,75.12,73.00,77.01,1.03
7,N. Milenković,traditional_18,35,75.47,72.96,76.98,1.05


In [6]:
split_half[[
    'player_name', 'layout', 'appearance_count', 'split_half_status',
    'median_split_half_similarity', 'split_half_p05', 'split_half_p95'
]].round(2)

,player_name,layout,appearance_count,split_half_status,median_split_half_similarity,split_half_p05,split_half_p95
0,Jair Cunha,positional_20,3,insufficient sample,NaN,NaN,NaN
1,Jair Cunha,traditional_18,3,insufficient sample,NaN,NaN,NaN
2,Morato,positional_20,11,calculated,86.79,83.11,91.96
3,Morato,traditional_18,11,calculated,86.59,81.25,90.66
4,Murillo,positional_20,23,calculated,90.46,86.32,93.60
5,Murillo,traditional_18,23,calculated,90.96,87.02,93.85
6,N. Milenković,positional_20,35,calculated,91.86,88.05,94.75
7,N. Milenković,traditional_18,35,calculated,91.38,87.52,94.39


## 4. Zone-layout sensitivity

The league-wide correlations show whether players receive similar scores and rankings under both layouts. High correlation supports the conclusion that the main result is not an artefact of one pitch division.

In [7]:
layout_robustness.round(3)

,population,player_count,pearson_score_correlation,spearman_rank_correlation,mean_absolute_score_difference,median_absolute_score_difference,p90_absolute_score_difference,maximum_absolute_score_difference
0,all player profiles,103,0.979,0.977,1.535,1.268,3.147,8.694
1,template-qualified players,78,0.983,0.974,1.297,1.111,2.732,4.502


In [8]:
summary.pivot(index='player_name', columns='layout', values='similarity_score').assign(
    absolute_difference=lambda frame: (frame['positional_20'] - frame['traditional_18']).abs()
).round(2)

layout,positional_20,traditional_18,absolute_difference
player_name,,,
Jair Cunha,57.92,53.74,4.18
Morato,73.64,72.24,1.40
Murillo,69.62,68.01,1.61
N. Milenković,75.12,75.47,0.35


## 5. Action-family sensitivity

Pass and defensive-action profiles answer different football questions. This check demonstrates how much the combined direct-action result depends on the high volume of passes; it should not be read as a performance ranking.

In [9]:
action_sensitivity.pivot_table(
    index=['player_name', 'layout'],
    columns='action_family',
    values=['similarity_score', 'family_action_count'],
).round(2)

family_action_count                            \
action_family                 all_direct_actions defensive_actions  passes   
player_name   layout                                                         
Jair Cunha    positional_20                161.0              38.0   115.0   
              traditional_18               161.0              38.0   115.0   
Morato        positional_20                856.0             185.0   620.0   
              traditional_18               856.0             185.0   620.0   
Murillo       positional_20               1599.0             384.0  1116.0   
              traditional_18              1599.0             384.0  1116.0   
N. Milenković positional_20               2088.0             549.0  1438.0   
              traditional_18              2088.0             549.0  1438.0   

                               similarity_score                           
action_family                all_direct_actions defensive_actions passes  
player_name   layout                                                      
Jair Cunha    positional_20               57.92             62.60  55.14  
              traditional_18              53.74             60.49  51.36  
Morato        positional_20               73.64             79.97  69.55  
              traditional_18              72.24             76.78  69.56  
Murillo       positional_20               69.62             80.94  63.99  
              traditional_18              68.01             80.08  63.68  
N. Milenković positional_20               75.12             84.74  71.28  
              traditional_18              75.47             82.36  70.97

## Interpretation and decision

The Forest maps occupy recognisable centre-back areas, while their lateral shifts agree with the decoded provider roles. The two layouts produce very similar league-wide scores and rankings, so the result is not materially dependent on choosing 18 rather than 20 zones. Murillo, Morato and Nikola Milenković have sufficiently stable samples for interpretation; Jair Cunha must remain provisional because his three appearances produce wide uncertainty and cannot support split-half validation.

The combined map is useful as a broad action-location profile, but the pass-only and defensive-action results differ enough to justify keeping action-family views separate in later scouting work. Similarity describes conventionality relative to league centre-backs, not player quality or tactical suitability.

**Decision:** the centre-back pipeline is sufficiently coherent to use as the reference implementation for extending the method to full-backs and wing-backs, provided that provisional samples remain visibly flagged and action-family profiles are retained.